## Assignment #1
**Author:** Nataliia Kobrii (UTORid: qq566325)

In [1]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Markdown reference can be found here:
    http://nestacms.com/docs/creating-content/markdown-cheat-sheet

Q1. Why would it be a problem if our training set and test set are the same.

<pre> If the training set and test set are the same, the model would simply memorize the data rather than learning 
generalizable patterns. This means we would not be able to evaluate how well the model performs on unseen data. 
The model would appear to have perfect or near-perfect performance (overfitting), but in reality it would 
likely perform poorly on new, unseen data points. The purpose of having a separate test set is to simulate 
real-world conditions where the model encounters data it has never seen before. </pre>

Q2 [OPTIONAL]. Explain step by step the process to select k in the k-nearest neighbor algorithm (pseudocode) 

1. Split the dataset into training and validation sets (or use k-fold cross-validation).
2. Choose a range of k values to evaluate (e.g., k = 1, 2, 3, ..., 20).
3. For each k value:
   - Train KNN on the training set with the current k.
   - Evaluate performance on the validation set (e.g., using accuracy or RMSE).
   - Record the validation score.
4. Compare validation scores across all k values.
5. Select the k that yields the best validation performance.
6. Optionally, plot validation score vs. k to visually identify the optimal value.

Note: Very small k values tend to overfit (high variance), while very large k values tend to underfit (high bias).

Q3. For the k-nearest regression. What happens when k = n. Where n is equal to the training size.

<pre> When k equals the entire training set size n, every training sample is considered a neighbor for any query point.
The prediction becomes the global mean of all training labels, regardless of where the query point is located. 
This results in a constant model that ignores the input features entirely, leading to severe underfitting 
and high bias. The model loses all ability to capture local patterns in the data. </pre>

Q4. Define a function that takes a 1-d numpy array, a parameter k, and a number p.
The function returns an estimate equal to the mean of the closest k points to the number p

In [2]:
def k_neighbor(input_data, k, p):
    """Returns the k-neighbor estimate for p using data input_data.
    
    Keyword arguments:
    input_data -- numpy array of all the data
    k -- Number of nearest neighbors
    p -- query point value
    
    Assumptions:
    - Ties are broken by selecting the value that appears first in the original array (stable sort).
    """
    
    # Compute absolute distances from each point to p
    distances = np.abs(input_data - p)
    print("Absolute differences:", distances)
    
    # Get indices of the k smallest distances (stable sort preserves order for ties)
    nearest_indices = np.argsort(distances, kind="stable")[:k]
    print("Indices of smallest distances:", nearest_indices)
    
    # Retrieve the k nearest values
    nearest_values = input_data[nearest_indices]
    print("K nearest values:", nearest_values)
    
    # Return the mean of the k nearest values  
    return np.mean(nearest_values)

In [3]:
# Evaluate
data = np.array([1, 3, 4, 5, 7, 8, 11, 12, 13, 15, 19, 24, 25, 29, 40])
print(k_neighbor(input_data=data, k=3, p=5))
print(k_neighbor(input_data=data, k=2, p=15))
print(k_neighbor(input_data=data, k=3, p=25))
print(k_neighbor(input_data=data, k=1, p=55))
print(k_neighbor(input_data=data, k=3, p=55))
print(k_neighbor(input_data=data, k=10, p=55))

Absolute differences: [ 4  2  1  0  2  3  6  7  8 10 14 19 20 24 35]
Indices of smallest distances: [3 2 1]
K nearest values: [5 4 3]
4.0
Absolute differences: [14 12 11 10  8  7  4  3  2  0  4  9 10 14 25]
Indices of smallest distances: [9 8]
K nearest values: [15 13]
14.0
Absolute differences: [24 22 21 20 18 17 14 13 12 10  6  1  0  4 15]
Indices of smallest distances: [12 11 13]
K nearest values: [25 24 29]
26.0
Absolute differences: [54 52 51 50 48 47 44 43 42 40 36 31 30 26 15]
Indices of smallest distances: [14]
K nearest values: [40]
40.0
Absolute differences: [54 52 51 50 48 47 44 43 42 40 36 31 30 26 15]
Indices of smallest distances: [14 13 12]
K nearest values: [40 29 25]
31.333333333333332
Absolute differences: [54 52 51 50 48 47 44 43 42 40 36 31 30 26 15]
Indices of smallest distances: [14 13 12 11 10  9  8  7  6  5]
K nearest values: [40 29 25 24 19 15 13 12 11  8]
19.6


In [4]:
# Observations:
# When p falls within the range of the data, the k-nearest neighbor estimate gives a reasonable local average.
# However, when p is far outside the data range (e.g., p=55), the estimate is pulled toward the largest values
# in the dataset and may not be a meaningful prediction. Increasing k for such outlier queries helps
# slightly by averaging over more points, but the estimate is still constrained by the data range.

Q5. Similar to Q4 but for the n dimensional case. 

In [5]:
def l1_norm(a, b):
    """Returns the L1 norm (Manhattan distance) between vectors a and b."""
    return np.sum(np.abs(a - b))

def l2_norm(a, b):
    """Returns the L2 norm (Euclidean distance) between vectors a and b."""
    return np.sqrt(np.sum((a - b) ** 2))

def k_neighbor_nd(input_data, k, p, metric='l1', mode='mean'):
    """Returns the k-neighbor estimate for point p using n-dimensional input_data.
    
    Keyword arguments:
    input_data -- numpy array of shape (n_samples, n_features)
    k -- number of nearest neighbors
    p -- query point (array-like)
    metric -- 'l1' (Manhattan) or 'l2' (Euclidean) distance
    mode -- aggregation method: 'mean', 'median', or 'max'
    
    Returns:
    float -- aggregated value of the k smallest distances
    """
    p = np.array(p)
    
    # Compute distance from query point to each data point
    if metric == 'l1':
        distances = np.array([l1_norm(p, x) for x in input_data])
    elif metric == 'l2':
        distances = np.array([l2_norm(p, x) for x in input_data])
    else:
        raise ValueError("Unsupported metric. Use 'l1' or 'l2'.")
    
    # Get k smallest distances
    k_distances = np.sort(distances)[:k]
    
    # Aggregate
    if mode == 'mean':
        return np.mean(k_distances)
    elif mode == 'median':
        return np.median(k_distances)
    elif mode == 'max':
        return np.max(k_distances)
    else:
        raise ValueError("Unsupported mode. Use 'mean', 'median', or 'max'.")

In [6]:
data_4d = np.array([[4, 1, 2, 1], [1, 4, 2, 0], [3, 3, 1, 1], 
        [4, 0, 0, 0], [1, 2, 0, 0], [3, 4, 2, 3], 
        [2, 4, 4, 2], [2, 1, 4, 1], [3, 3, 2, 4], 
        [4, 3, 0, 4], [2, 2, 4, 0], [4, 3, 0, 2], 
        [4, 3, 0, 2], [0, 3, 4, 2]])

In [7]:
# Evaluate with L1 norm
print("L1 Norm Results:")
print(k_neighbor_nd(input_data=data_4d, k=3, p=[2, 1, 4, 3], metric='l1', mode='mean'))
print(k_neighbor_nd(input_data=data_4d, k=2, p=[4, 4, 0, 0], metric='l1', mode='mean'))
print(k_neighbor_nd(input_data=data_4d, k=3, p=[2, 2, 2, 4], metric='l1', mode='max'))
print(k_neighbor_nd(input_data=data_4d, k=1, p=[2, 3, 3, 4], metric='l1', mode='mean'))
print(k_neighbor_nd(input_data=data_4d, k=3, p=[2, 3, 3, 4], metric='l1', mode='median'))

# Evaluate with L2 norm
print("\nL2 Norm Results:")
print(k_neighbor_nd(input_data=data_4d, k=3, p=[2, 1, 4, 3], metric='l2', mode='mean'))
print(k_neighbor_nd(input_data=data_4d, k=2, p=[4, 4, 0, 0], metric='l2', mode='mean'))
print(k_neighbor_nd(input_data=data_4d, k=3, p=[2, 2, 2, 4], metric='l2', mode='max'))
print(k_neighbor_nd(input_data=data_4d, k=1, p=[2, 3, 3, 4], metric='l2', mode='mean'))
print(k_neighbor_nd(input_data=data_4d, k=3, p=[2, 3, 3, 4], metric='l2', mode='median'))

L1 Norm Results:
3.3333333333333335
3.0
5
2.0
4.0

L2 Norm Results:
2.720759220056127
2.118033988749895
3.0
1.4142135623730951
2.0


In [8]:
# Observations:
# L1 and L2 norms produce different distance values, which can lead to different neighbor selections.
# L1 (Manhattan) sums absolute differences along each dimension, while L2 (Euclidean) penalizes
# large differences in any single dimension more heavily due to squaring.
# When k=1, both metrics return the same nearest neighbor in most cases, but diverge as k increases.

Q6[Optional]. Read the documentation on KNeighborsRegressor

http://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html
    
Explore the source code:
    https://github.com/scikit-learn/scikit-learn/blob/ef5cb84a/sklearn/neighbors/regression.py
        
How different it is from your implementation? How well can you follow the code? Did you learn something new?

<pre> The scikit-learn KNeighborsRegressor is much more comprehensive than my implementation. It uses efficient 
data structures like KD-trees and Ball trees for fast neighbor lookup, supports multiple distance metrics, 
and includes both uniform and distance-based weighting. My implementation uses a brute-force approach, 
computing distances to all points. The sklearn version also follows the fit/predict paradigm where the 
model is first trained on data and then used for predictions, whereas mine computes everything in one call. </pre>